# Tabular GAN через SDV

Минимальный скелет ноутбука: загрузить таблицу в `df`, убрать лишние колонки, обучить `CTGANSynthesizer`, получить `synthetic_df`.

## Зависимости

```bash
uv add sdv
```

In [ ]:
from pathlib import Path

import pandas as pd
from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer

## Загрузка таблицы

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TABLE_NAME = "table"
TABLE_PATH = PROJECT_ROOT / "data" / "table.csv"

In [ ]:
if TABLE_PATH.suffix.lower() in {".xlsx", ".xls"}:
    df = pd.read_excel(TABLE_PATH)
else:
    df = pd.read_csv(TABLE_PATH)

display(df.head())
df.shape

## Подготовка

In [ ]:
DROP_COLUMNS = []

df_model = df.drop(columns=DROP_COLUMNS, errors="ignore").copy()
df_model.shape

## Конфиг обучения

In [ ]:
CTGAN_PARAMS = {
    "epochs": 100,
    "batch_size": 500,
    "pac": 10,
    "embedding_dim": 128,
    "generator_dim": (256, 256),
    "discriminator_dim": (256, 256),
    "generator_lr": 2e-4,
    "discriminator_lr": 2e-4,
    "discriminator_steps": 1,
    "enforce_min_max_values": True,
    "enforce_rounding": True,
    "enable_gpu": True,
    "verbose": True,
}

## Обучение

In [ ]:
metadata = Metadata.detect_from_dataframe(
    data=df_model,
    table_name=TABLE_NAME,
)

synthesizer = CTGANSynthesizer(
    metadata=metadata,
    **CTGAN_PARAMS,
)
synthesizer.fit(df_model)

## Генерация

In [ ]:
SYNTHETIC_ROWS = len(df_model)

synthetic_df = synthesizer.sample(num_rows=SYNTHETIC_ROWS)
display(synthetic_df.head())
synthetic_df.shape